In [1]:
from calendar import month_name
from unicodedata import category

import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
import urllib.parse

In [2]:
load_dotenv()
def get_db_connection():
    
    user = os.getenv('MYSQL_USER')
    password = os.getenv('MYSQL_PASSWORD')
    host = os.getenv('MYSQL_HOST')
    port = os.getenv('MYSQL_PORT')
    database = os.getenv('MYSQL_DB')

    safe_password = urllib.parse.quote_plus(password)

    db_url = f"mysql+pymysql://{user}:{safe_password}@{host}:{port}/{database}"
    engine = create_engine(db_url)
    return engine

def query_data(query):
    try:
        engine = get_db_connection()
        df_result = pd.read_sql(query,con=engine)
        return df_result
    except Exception as e:
        print(f"Error in querying data: {e}")
        return None

In [3]:
household_2024_query = """
select exp.newid,exp.reference_month,exp.reference_year,exp.ucc,exp.cost,household.fam_size,household.family_income_before_tax_last_12_month,household.calibration_weight,household.number_of_earners,household.popsize,household.interview_month,household.interview_year,household.region,household.sex_ref,household.state,household.psu,household.division,household.total_salary_income_before_deduction from monthly_expenditure exp left join household_metadata household on exp.newid = household.newid where exp.ucc in ('690114','690111','690120','270102','270106','690113',
     '690114','690310','300311','300312','300321','300322','300331','300332','320522','320232','690117','690119','690116',
     '480100','480213','490501','310316','310140','270310','620930','310231','310232','310400','340610','340902','310314',
     '310350','610130','310243','620917','620918','310333','690320','690330','590230','690118','300311','300312','300321',
     '300322','320331','320332','320522','320232','690111','690117','690119','690120','690115','690116','690210','270106','690310',
     '620930','270310','310140','310231','310232','620917','620918','310243','310400','310316','310314','610130','310333','310350') and exp.reference_year in(2020, 2021,2022,2023,2023);

"""

df_exp_data = query_data(household_2024_query)

In [4]:
df_exp_data.head()

,newid,reference_month,reference_year,ucc,cost,fam_size,family_income_before_tax_last_12_month,calibration_weight,number_of_earners,popsize,interview_month,interview_year,region,sex_ref,state,psu,division,total_salary_income_before_deduction
0,4280803,2,2020,270310,6.0,2.0,34507.0,29066.090,0.0,2.0,5.0,2020.0,3.0,1.0,37.0,None,5.0,0.0
1,4349223,5,2020,270310,67.0,1.0,18535.0,29013.710,0.0,5.0,6.0,2020.0,3.0,2.0,54.0,None,5.0,0.0
2,4214874,1,2020,270102,260.0,2.0,1735.0,20883.818,0.0,3.0,4.0,2020.0,3.0,1.0,12.0,None,5.0,0.0
3,4214874,2,2020,270102,269.0,2.0,1735.0,20883.818,0.0,3.0,4.0,2020.0,3.0,1.0,12.0,None,5.0,0.0
4,4214874,3,2020,270102,260.0,2.0,1735.0,20883.818,0.0,3.0,4.0,2020.0,3.0,1.0,12.0,None,5.0,0.0


In [6]:
#df_2024_data.drop_duplicates(inplace=True)

#df_2024_data.dropna(inplace = True)

ucc_2024_map ={
    # Telecommunications
    270102: "Cellular phone service",
    270106: "Residential telephone including VOIP",
    270310: "Cable and satellite television services",
    690114: "Computer information services (internet)",
    690116: "Internet services away from home",
    
    # Computing Hardware & Accessories
    690111: "Computers and computer hardware for nonbusiness use",
    690117: "Portable memory",
    690120: "Computer accessories",
    690115: "Personal digital assistants",
    320232: "Telephones and accessories",
    690210: "Telephone answering devices",
    
    # Software & Digital Services
    690119: "Computer software",
    620930: "Online gaming services",
    310400: "Applications, games, and ringtones for handheld devices",
    
    # Computing Services
    690113: "Repair of computer systems for nonbusiness use",
    690310: "Installation of computers",
    
    # Streaming & Digital Media
    310350: "Streaming and downloading audio",
    310243: "Rental, streaming, and downloading videos",
    620917: "Rental of video hardware/accessories",
    620918: "Rental of video software",
    
    # Gaming
    310231: "Video game software",
    310232: "Video game hardware and accessories",
    
    # Audio/Visual Equipment
    310140: "Televisions",
    310316: "Stereos, radios, speakers, and sound components",
    310314: "Personal digital audio players",
    310333: "Accessories and other sound equipment",
    340610: "Repair of televisions, radio, and sound equipment",
    340902: "Rental of televisions",
    690320: "Installation of televisions",
    690330: "Installation of satellite television equipment",
    
    # Musical Instruments
    610130: "Musical instruments and accessories",
    
    # Digital Reading
    590230: "Books, digital books, or book subscriptions",
    690118: "Digital book readers",
    
    # Appliances (Non-Digital - appear to be duplicates/errors in original list)
    300311: "Cooking stoves and ovens (renter)",
    300312: "Cooking stoves and ovens (owned home)",
    300321: "Microwave ovens (renter)",
    300322: "Microwave ovens (owned home)",
    300331: "Portable dishwashers (renter)",
    300332: "Portable dishwashers (owned home)",
    320522: "Portable heating and cooling equipment",
    
    # Vehicle Accessories (Non-Digital - appear to be errors in original list)
    480100: "Vehicle parts, accessories, fluid excluding tires",
    480213: "Parts, equipment, and accessories",
    490501: "Vehicle accessories including labor",

}
category_map = {
    # Online Service
    270102: "Online Service",   # Cellular phone service
    270106: "Online Service",   # Residential telephone/VOIP
    270310: "Online Service",   # Cable and satellite TV
    690114: "Online Service",   # Internet services
    690116: "Online Service",   # Internet away from home
    620930: "Online Service",   # Online gaming
    310350: "Online Service",   # Streaming audio
    310243: "Online Service",   # Streaming/downloading video
    590230: "Online Service",   # Digital books/subscriptions

    # Software
    690119: "Software",         # Computer software
    310400: "Software",         # Apps, games, ringtones
    310231: "Software",         # Video game software
    620917: "Software",         # Rental of video software
    620918: "Software",         # Rental of video software

    # Electronics
    690111: "Electronics",      # Computers and hardware
    690117: "Electronics",      # Portable memory
    690120: "Electronics",      # Computer accessories
    690115: "Electronics",      # PDAs
    320232: "Electronics",      # Telephones and accessories
    690210: "Electronics",      # Telephone answering devices
    690113: "Electronics",      # Computer repair
    690310: "Electronics",      # Computer installation
    310232: "Electronics",      # Video game hardware
    310140: "Electronics",      # Televisions
    310316: "Electronics",      # Stereos, speakers
    310314: "Electronics",      # Personal audio players
    310333: "Electronics",      # Sound accessories
    340610: "Electronics",      # Repair of AV equipment
    340902: "Electronics",      # Rental of televisions
    690320: "Electronics",      # Installation of televisions
    690330: "Electronics",      # Installation of satellite equipment
    690118: "Electronics",      # Digital book readers
    610130: "Electronics",      # Musical instruments
    300311: "Electronics",      # Stoves/ovens (renter)
    300312: "Electronics",      # Stoves/ovens (owned)
    300321: "Electronics",      # Microwaves (renter)
    300322: "Electronics",      # Microwaves (owned)
    300331: "Electronics",      # Dishwashers (renter)
    300332: "Electronics",      # Dishwashers (owned)
    320522: "Electronics",      # Portable heating/cooling
    480100: "Electronics",      # Vehicle parts/accessories
    480213: "Electronics",      # Parts and equipment
    490501: "Electronics",      # Vehicle accessories
}

df_exp_data['product_description'] = df_exp_data['ucc'].map(ucc_2024_map).fillna(df_exp_data['ucc'])

#df_2024_data.drop(columns=['ALCNO','PUBFLAG','UCCSEQ'],inplace=True)

df_exp_data['product_category'] = df_exp_data['ucc'].map(category_map)

df_exp_data.head()

,newid,reference_month,reference_year,ucc,cost,fam_size,family_income_before_tax_last_12_month,calibration_weight,number_of_earners,popsize,interview_month,interview_year,region,sex_ref,state,psu,division,total_salary_income_before_deduction,product_description,product_category
0,4280803,2,2020,270310,6.0,2.0,34507.0,29066.090,0.0,2.0,5.0,2020.0,3.0,1.0,37.0,None,5.0,0.0,Cable and satellite television services,Online Service
1,4349223,5,2020,270310,67.0,1.0,18535.0,29013.710,0.0,5.0,6.0,2020.0,3.0,2.0,54.0,None,5.0,0.0,Cable and satellite television services,Online Service
2,4214874,1,2020,270102,260.0,2.0,1735.0,20883.818,0.0,3.0,4.0,2020.0,3.0,1.0,12.0,None,5.0,0.0,Cellular phone service,Online Service
3,4214874,2,2020,270102,269.0,2.0,1735.0,20883.818,0.0,3.0,4.0,2020.0,3.0,1.0,12.0,None,5.0,0.0,Cellular phone service,Online Service
4,4214874,3,2020,270102,260.0,2.0,1735.0,20883.818,0.0,3.0,4.0,2020.0,3.0,1.0,12.0,None,5.0,0.0,Cellular phone service,Online Service


In [8]:
df_exp_data.describe()
df_exp_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 869484 entries, 0 to 869483
Data columns (total 20 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   newid                                   869484 non-null  int64  
 1   reference_month                         869484 non-null  int64  
 2   reference_year                          869484 non-null  int64  
 3   ucc                                     869484 non-null  int64  
 4   cost                                    869484 non-null  float64
 5   fam_size                                675566 non-null  float64
 6   family_income_before_tax_last_12_month  675566 non-null  float64
 7   calibration_weight                      675566 non-null  float64
 8   number_of_earners                       675566 non-null  float64
 9   popsize                                 675566 non-null  float64
 10  interview_month                         6755

In [10]:
df_exp_data.isna().sum()
df_exp_data.dropna(inplace=True)

In [24]:
df_exp_data.duplicated().sum()
df_exp_data.drop_duplicates(inplace=True)
df_exp_data.duplicated().sum()
df_exp_data.head()

,newid,reference_month,reference_year,ucc,cost,fam_size,family_income_before_tax_last_12_month,calibration_weight,number_of_earners,popsize,interview_month,interview_year,region,sex_ref,state,psu,division,total_salary_income_before_deduction,product_description,product_category
29,4214954,1,2020,270102,45.0,1.0,10000.0,23581.165,1.0,2.0,4.0,2020.0,4.0,1.0,6.0,S49C,9.0,5000.0,Cellular phone service,Online Service
30,4214954,2,2020,270102,45.0,1.0,10000.0,23581.165,1.0,2.0,4.0,2020.0,4.0,1.0,6.0,S49C,9.0,5000.0,Cellular phone service,Online Service
31,4214954,3,2020,270102,45.0,1.0,10000.0,23581.165,1.0,2.0,4.0,2020.0,4.0,1.0,6.0,S49C,9.0,5000.0,Cellular phone service,Online Service
32,4214954,1,2020,690114,120.0,1.0,10000.0,23581.165,1.0,2.0,4.0,2020.0,4.0,1.0,6.0,S49C,9.0,5000.0,Computer information services (internet),Online Service
33,4214954,2,2020,690114,120.0,1.0,10000.0,23581.165,1.0,2.0,4.0,2020.0,4.0,1.0,6.0,S49C,9.0,5000.0,Computer information services (internet),Online Service


In [27]:
df_exp_data['cost']= df_exp_data['cost'].astype("int64")
df_exp_data['fam_size']= df_exp_data['fam_size'].astype("int64")
df_exp_data['family_income_before_tax_last_12_month']= df_exp_data['family_income_before_tax_last_12_month'].astype("int64")
df_exp_data['number_of_earners']= df_exp_data['number_of_earners'].astype("int64")
df_exp_data['popsize']= df_exp_data['popsize'].astype("int64")
df_exp_data['interview_month']= df_exp_data['interview_month'].astype("int64")
df_exp_data['interview_year']= df_exp_data['interview_year'].astype("int64")
df_exp_data['region']= df_exp_data['region'].astype("int64")
df_exp_data['sex_ref']= df_exp_data['sex_ref'].astype("int64")
df_exp_data['state']= df_exp_data['state'].astype("int64")
df_exp_data['division']= df_exp_data['division'].astype("int64")
df_exp_data['total_salary_income_before_deduction']= df_exp_data['total_salary_income_before_deduction'].astype("int64")
df_exp_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 262467 entries, 29 to 653400
Data columns (total 20 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   newid                                   262467 non-null  int64  
 1   reference_month                         262467 non-null  int64  
 2   reference_year                          262467 non-null  int64  
 3   ucc                                     262467 non-null  int64  
 4   cost                                    262467 non-null  int64  
 5   fam_size                                262467 non-null  int64  
 6   family_income_before_tax_last_12_month  262467 non-null  int64  
 7   calibration_weight                      262467 non-null  float64
 8   number_of_earners                       262467 non-null  int64  
 9   popsize                                 262467 non-null  int64  
 10  interview_month                         262467 n

In [28]:
region_dict ={
    1: "Northeast",
    2: "Midwest",
    3: "South",
    4: "West"
}
df_exp_data['region'] = df_exp_data['region'].map(region_dict).fillna(df_exp_data['region'])
df_exp_data.head()

,newid,reference_month,reference_year,ucc,cost,fam_size,family_income_before_tax_last_12_month,calibration_weight,number_of_earners,popsize,interview_month,interview_year,region,sex_ref,state,psu,division,total_salary_income_before_deduction,product_description,product_category
29,4214954,1,2020,270102,45,1,10000,23581.165,1,2,4,2020,West,1,6,S49C,9,5000,Cellular phone service,Online Service
30,4214954,2,2020,270102,45,1,10000,23581.165,1,2,4,2020,West,1,6,S49C,9,5000,Cellular phone service,Online Service
31,4214954,3,2020,270102,45,1,10000,23581.165,1,2,4,2020,West,1,6,S49C,9,5000,Cellular phone service,Online Service
32,4214954,1,2020,690114,120,1,10000,23581.165,1,2,4,2020,West,1,6,S49C,9,5000,Computer information services (internet),Online Service
33,4214954,2,2020,690114,120,1,10000,23581.165,1,2,4,2020,West,1,6,S49C,9,5000,Computer information services (internet),Online Service


In [ ]:
population_dict ={
    1: "5+ Million",
    2: "1-5 Million",
    3: "0.5-1.0 Million",
    4: "100-500 Thousands",
    5: "100- Thousands",
    6: "Suppressed"
}
df_exp_data['population_size'] = df_exp_data['popsize'].map(population_dict)


In [40]:
df_exp_data.head()

,newid,reference_month,reference_year,ucc,cost,fam_size,family_income_before_tax_last_12_month,calibration_weight,number_of_earners,interview_month,interview_year,region,sex_ref,state,psu,division,total_salary_income_before_deduction,product_description,product_category,population_size
29,4214954,1,2020,270102,45,1,10000,23581.165,1,4,2020,West,1,6,S49C,9,5000,Cellular phone service,Online Service,1-5 Million
30,4214954,2,2020,270102,45,1,10000,23581.165,1,4,2020,West,1,6,S49C,9,5000,Cellular phone service,Online Service,1-5 Million
31,4214954,3,2020,270102,45,1,10000,23581.165,1,4,2020,West,1,6,S49C,9,5000,Cellular phone service,Online Service,1-5 Million
32,4214954,1,2020,690114,120,1,10000,23581.165,1,4,2020,West,1,6,S49C,9,5000,Computer information services (internet),Online Service,1-5 Million
33,4214954,2,2020,690114,120,1,10000,23581.165,1,4,2020,West,1,6,S49C,9,5000,Computer information services (internet),Online Service,1-5 Million


In [42]:
state_mapping = {
    1: "Alabama",
    2: "Alaska",
    4: "Arizona",
    5: "Arkansas",
    6: "California",
    8: "Colorado",
    9: "Connecticut",
    10: "Delaware",
    11: "District of Columbia",
    12: "Florida",
    13: "Georgia",
    14: "Massachusetts",
    15: "Hawaii",
    16: "Idaho",
    17: "Illinois",
    18: "Indiana",
    19: "Iowa",
    20: "Kansas",
    21: "Kentucky",
    22: "Louisiana",
    23: "Maine",
    24: "Maryland",
    25: "Massachusetts",
    26: "Michigan",
    27: "Minnesota",
    28: "Mississippi",
    29: "Missouri",
    30: "Montana",
    31: "Nebraska",
    32: "Nevada",
    33: "New Hampshire",
    34: "New Jersey",
    35: "New Mexico",
    36: "New York",
    37: "North Carolina",
    39: "Ohio",
    40: "Oklahoma",
    41: "Oregon",
    42: "Pennsylvania",
    44: "Rhode Island",
    45: "South Carolina",
    46: "South Dakota",
    47: "Tennessee",
    48: "Texas",
    49: "Utah",
    51: "Virginia",
    52: "Maryland",
    53: "Washington",
    54: "West Virginia",
    55: "Wisconsin",
    72: "Puerto Rico"
}
df_exp_data['state'] = df_exp_data['state'].map(state_mapping).fillna(df_exp_data['state'])
print(df_exp_data['state'].head())

29    California
30    California
31    California
32    California
33    California
Name: state, dtype: object


In [43]:
print(df_exp_data.head(10))

      newid  reference_month  reference_year     ucc  cost  fam_size  \
29  4214954                1            2020  270102    45         1   
30  4214954                2            2020  270102    45         1   
31  4214954                3            2020  270102    45         1   
32  4214954                1            2020  690114   120         1   
33  4214954                2            2020  690114   120         1   
34  4214954                3            2020  690114   120         1   
35  4214964                1            2020  270102    56         2   
36  4214964                2            2020  270102    56         2   
37  4214964                3            2020  270102    56         2   
38  4214974                1            2020  270102    48         2   

    family_income_before_tax_last_12_month  calibration_weight  \
29                                   10000           23581.165   
30                                   10000           23581.165   
31       

In [44]:
division_dict ={
    1: "New England",
    2: "Middle Atlantic",
    3: "East North Central",
    4: "West North Central",
    5: "South Central",
    6: "East South Central",
    7: "West South Central",
    8: "Mountain",
    9: "Pacific"
}

df_exp_data['division'] = df_exp_data['division'].map(division_dict).fillna(df_exp_data['division'])
print(df_exp_data['division'].head())

29    Pacific
30    Pacific
31    Pacific
32    Pacific
33    Pacific
Name: division, dtype: object


In [45]:
print(df_exp_data.head(10))

      newid  reference_month  reference_year     ucc  cost  fam_size  \
29  4214954                1            2020  270102    45         1   
30  4214954                2            2020  270102    45         1   
31  4214954                3            2020  270102    45         1   
32  4214954                1            2020  690114   120         1   
33  4214954                2            2020  690114   120         1   
34  4214954                3            2020  690114   120         1   
35  4214964                1            2020  270102    56         2   
36  4214964                2            2020  270102    56         2   
37  4214964                3            2020  270102    56         2   
38  4214974                1            2020  270102    48         2   

    family_income_before_tax_last_12_month  calibration_weight  \
29                                   10000           23581.165   
30                                   10000           23581.165   
31       

In [46]:
psu_dict ={
    1102: "Philadelphia – Wilmington – Atlantic City, PA – NJ – DE - MD",
    1103: "Boston – Brockton – Nashua, MA – NH – ME CT",
    1109: "New York, NY",
    1110: "New York, Connecticut suburbs",
    1111: "New Jersey suburbs",
    1207: "Chicago – Gary – Kenosha, IL – IN - WI",
    1208: "Detroit – Ann Arbor – Flint, MI",
    1210: "Cleveland – Akron, OH",
    1211: "Minneapolis – St. Paul, MN – WI",
    1312: "Washington, DC – MD – VA – WV",
    1313: "Baltimore, MD",
    1316: "Dallas – Ft. Worth, TX",
    1318: "Houston – Galveston – Brazoria, TX",
    1319: "Atlanta, GA",
    1320: "Miami – Ft. Lauderdale, FL",
    1419: "Los Angeles – Orange, CA",
    1420: "Los Angeles suburbs, CA",
    1422: "San Francisco – Oakland – San Jose, CA",
    1423: "Seattle – Tacoma – Bremerton, WA",
    1424: "San Diego, CA",
    1429: "Phoenix – Mesa, AZ",
    "S11A": "Boston-Cambridge-Newton, MA-NH",
    "S12A": "New York-Newark-Jersey City, NY-NJ-PA",
    "S12B": "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",
    "S23A": "Chicago-Naperville-Elgin, IL-IN-WI",
    "S23B": "Detroit-Warren-Dearborn, MI",
    "S24A": "Minneapolis-St. Paul-Bloomington, MN-WI",
    "S24B": "St. Louis, MO-IL",
    "S35A": "Washington-Arlington-Alexandria, DC-VA-MD-WV",
    "S35B": "Miami-Fort Lauderdale-West Palm Beach, FL",
    "S35C": "Atlanta-Sandy Springs-Roswell, GA",
    "S35D": "Tampa-St. Petersburg-Clearwater, FL",
    "S35E": "Baltimore-Columbia-Towson, MD",
    "S37A": "Dallas-Fort Worth-Arlington, TX",
    "S37B": "Houston-The Woodlands-Sugar Land, TX",
    "S48A": "Phoenix-Mesa-Scottsdale, AZ",
    "S48B": "Denver-Aurora-Lakewood, CO",
    "S49A": "Los Angeles-Long Beach-Anaheim, CA",
    "S49B": "San Francisco-Oakland-Hayward, CA",
    "S49C": "Riverside-San Bernardino-Ontario, CA",
    "S49D": "Seattle-Tacoma-Bellevue, WA",
    "S49E": "San Diego-Carlsbad, CA",
    "S49F": "Honolulu, HI",
    "S49G": "Anchorage, AK"

}

df_exp_data['psu'] = df_exp_data['psu'].map(psu_dict).fillna(df_exp_data['psu'])

In [47]:
print(df_exp_data.loc[df_exp_data['state'] == "New York"])

          newid  reference_month  reference_year     ucc  cost  fam_size  \
35      4214964                1            2020  270102    56         2   
36      4214964                2            2020  270102    56         2   
37      4214964                3            2020  270102    56         2   
116     4215264                1            2020  270106    94         1   
117     4215264                2            2020  270106    94         1   
...         ...              ...             ...     ...   ...       ...   
610444  5596221               12            2023  310243    23         2   
610446  5596221               11            2023  310243    23         2   
611200  5606271               12            2023  310243    12         1   
633996  5543222               10            2023  690114   302         3   
634000  5543222               12            2023  690114   308         3   

        family_income_before_tax_last_12_month  calibration_weight  \
35               

In [48]:
sex_map ={
    1: "Male",
    2: "Female"
}

df_exp_data['sex_ref'] = df_exp_data['sex_ref'].map(sex_map).fillna(df_exp_data['sex_ref'])

In [49]:
df_exp_data.describe()

,newid,reference_month,reference_year,ucc,cost,fam_size,family_income_before_tax_last_12_month,calibration_weight,number_of_earners,interview_month,interview_year,total_salary_income_before_deduction
count,2.624670e+05,262467.000000,262467.000000,262467.000000,262467.000000,262467.000000,2.624670e+05,262467.000000,262467.000000,262467.000000,262467.000000,262467.000000
mean,4.820565e+06,6.684231,2021.338759,406681.645247,91.998175,2.542769,1.097922e+05,22541.707382,1.412543,6.529674,2021.519650,98191.838444
std,4.017315e+05,3.411492,1.210964,184855.632877,172.076713,1.476145,1.149796e+05,10592.119295,1.001616,3.447811,1.277782,105211.916761
min,4.214954e+06,1.000000,2020.000000,270102.000000,0.000000,1.000000,-1.049990e+05,940.429000,0.000000,1.000000,2020.000000,0.000000
25%,4.465701e+06,4.000000,2020.000000,270102.000000,30.000000,1.000000,3.210600e+04,15392.234500,1.000000,4.000000,2020.000000,12000.000000
50%,4.748351e+06,7.000000,2021.000000,310243.000000,61.000000,2.000000,7.706500e+04,22167.583000,1.000000,7.000000,2021.000000,70000.000000
75%,5.252903e+06,10.000000,2023.000000,690114.000000,102.000000,3.000000,1.500000e+05,28756.607000,2.000000,9.000000,2023.000000,140800.000000
max,5.607981e+06,12.000000,2023.000000,690330.000000,17000.000000,16.000000,1.221167e+06,107849.201000,8.000000,12.000000,2024.000000,788090.000000


In [50]:
df_exp_data.columns

Index(['newid', 'reference_month', 'reference_year', 'ucc', 'cost', 'fam_size',
       'family_income_before_tax_last_12_month', 'calibration_weight',
       'number_of_earners', 'interview_month', 'interview_year', 'region',
       'sex_ref', 'state', 'psu', 'division',
       'total_salary_income_before_deduction', 'product_description',
       'product_category', 'population_size'],
      dtype='object')

In [52]:
print(df_exp_data.tail())

          newid  reference_month  reference_year     ucc  cost  fam_size  \
646822  5582421               12            2023  690114   308         3   
649772  5589441               11            2023  690114   302         2   
649774  5589441               12            2023  690114   320         2   
650660  5592321               11            2023  270310   273         2   
653400  5606031               12            2023  690114   302         2   

        family_income_before_tax_last_12_month  calibration_weight  \
646822                                  364000           13218.766   
649772                                   60000            2235.675   
649774                                   60000            2235.675   
650660                                   47300           28040.543   
653400                                       0           16578.241   

        number_of_earners  interview_month  interview_year region sex_ref  \
646822                  3                1   

In [106]:
df_2024_data.to_csv('./cleaned_2024_digital_products_expenditure_dataset.csv')


In [111]:
df_2024 = pd.read_csv('cleaned_2024_digital_products_expenditure_dataset.csv')

In [55]:
month_map ={
    1 : "January",
    2 : "February",
    3 : "March",
    4 : "April",
    5 : "May",
    6 : "June",
    7 : "July",
    8 : "August",
    9 : "September",
    10 : "October",
    11 : "November",
    12 : "December"
}
df_exp_data['interview_month'] = df_exp_data['interview_month'].map(month_map).fillna(df_exp_data['interview_month'])
df_exp_data['reference_month'] = df_exp_data['reference_month'].map(month_map).fillna(df_exp_data['reference_month'])

In [56]:
print(df_exp_data.head())

      newid reference_month  reference_year     ucc  cost  fam_size  \
29  4214954         January            2020  270102    45         1   
30  4214954        February            2020  270102    45         1   
31  4214954           March            2020  270102    45         1   
32  4214954         January            2020  690114   120         1   
33  4214954        February            2020  690114   120         1   

    family_income_before_tax_last_12_month  calibration_weight  \
29                                   10000           23581.165   
30                                   10000           23581.165   
31                                   10000           23581.165   
32                                   10000           23581.165   
33                                   10000           23581.165   

    number_of_earners interview_month  interview_year region sex_ref  \
29                  1           April            2020   West    Male   
30                  1           